
Proposition 1: Suppose there exists a smooth feedback control law

$u = k(x, t) + v$ 

that makes the closed-loop system strictly contracting with rate $\lambda$ in some metric $M(x, t)$ for any piecewise-continuous signal $v(t)$. Then, for all $x, u, t$:

$ \dot{M} + (A + BK)'M + M(A + BK) < -2\lambda M \quad (4) $,

where $A = \frac{\partial f}{\partial x} + \sum_{i}^{m}\frac{\partial b_i}{\partial x} u_i$, $B = \frac{\partial f}{\partial u}$, and $K = \frac{\partial k}{\partial x}$.

Then this is also true: 

$\delta'_x MB = 0 \Rightarrow \delta'_x (\dot{M} + A'M + MA + 2\lambda M)\delta_x \leq 0. \quad (5)
$

(terms with $BK$ drop off due to being in the nullspace)

$(5)$ = Intrinsic contraction of the ACTUATED (A := A(x,u)) system in the directions where the system cannot actuate.
$(5)$ implies the existence of some stabilizing control for any target trajectory.

the whole implication in $(5)$ can be rewritten as:

$B_\perp'(-\dot{W} + AW + WA' + 2\lambda W)B_\perp < 0 \quad (6)$

where $W = M^{-1}$ 

and $(4)$ as:

$-\dot{W} + (A + BK)W + W(A + BK)' + 2\lambda W < 0 \quad (7)$

In [1]:
# contraction_condition = M_dot + (A + B@K).transpose(0,1)@M + M@(A + B@K) + 2 * self._lambda * M
def weak_contraction_condition(A, B, K, M, M_dot, lambda_):
    MABK = M@(A + B@K)
    result = M_dot + MABK.T + MABK + 2 * lambda_ * M
    return result
def weak_naturally_contracting_condition(A, W, W_dot, B_null, lambda_):
    # delta_x.T @ B @ M = 0 =>  directional_f_M + MA.T + MA + 2 * lambda_ * M < 0
    raise NotImplementedError()
    
def weak_inverse_metric_contraction_condition(A, B, K, W, W_dot, lambda_):
    ABKW = (A+B@K)@W
    result = (-W_dot + ABKW.T + ABKW + 2 * lambda_ * W)
    return result
def weak_inverse_metric_naturally_contracting_condition(A, W, W_dot, B_null, lambda_):
    AW = A@W
    result = B_null.T@(-W_dot + AW.T + AW  + 2 * lambda_ * W) @ B_null
    return result



Since the differential dynamics are linear, it is tempting to look for an admissible differential feedback controller of the form $\delta_u = K(x, t)\delta_x$ satisfying (4). We will show that this is possible under the following slightly stronger conditions:

$C_1$: if $\delta_x \neq 0$ satisfies $\delta_x' MB = 0 $, 

$\delta'_x MB = 0 \Rightarrow \delta_x' \left( \frac{\partial M}{\partial t} + \partial_f M + \frac{\partial f}{\partial x}' M + M \frac{\partial f}{\partial x} + 2\lambda M \right) \delta_x < 0 \quad (8)$

$C_2$: for each $i = 1, 2, \ldots, m $, \, $\partial_{b_i}M + \frac{\partial{b_i}}{\partial x}' M +  M
\frac{\partial{b_i}}{\partial x} = 0\quad (9)$.

Compared to $(5)$, $C_2$ makes $C_1$ drop off the terms related to $B$ (including  $\partial_{Bu} M$) and $\frac{\partial b_i}{\partial x}$.

$(8)$ gets rewritten using the dual metric as:

$B_\perp'(-\frac{\partial W}{\partial t} - \partial_f W +  \frac{\partial f}{\partial x}W + W \frac{\partial f}{\partial x}' + 2\lambda W)B_\perp < 0 \quad (6)$

and $(9)$ as:

$i = 1, 2, \ldots, m $, \, $\partial_{b_i}W - \frac{\partial{b_i}}{\partial x} W -  W
\frac{\partial{b_i}}{\partial x}' = 0\quad (10)$

Additionally:

$(9)$ holding means that $(4)$ is equal to:
$  \frac{\partial M}{\partial t} + \partial_f M + (\frac{\partial f}{\partial x} + BK)'M + M(\frac{\partial f}{\partial x} + BK) < -2\lambda M$

and $(7)$:


$-\frac{\partial W}{\partial t} - \partial_f W + (\frac{\partial f}{\partial x} + BK)W + W(\frac{\partial f}{\partial x} + BK)' + 2\lambda W < 0$

In [2]:

def simpler_controllers_contraction_conditions(partial_f_partial_x,B,K,M, directional_f_M, directional_B_M, partial_B_partial_x, lambda_, **kwargs):
    MdfdxBK = M@(partial_f_partial_x + B@K)
    C1 = (MdfdxBK + MdfdxBK.T + directional_f_M + 2 * lambda_ * M)
    
    C2s = []
    for i in range(0,B.shape[1]):
        b_i = B[:,i]
        partial_b_i_partial_x = partial_B_partial_x[:,i,:]
        directional_b_i_M = directional_B_M[:,:,i]
        Mdbdx = M@partial_b_i_partial_x
        C2_condition_i = (directional_b_i_M + Mdbdx + Mdbdx.T) # == 0 
        C2s.append(C2_condition_i)
    return C1,C2s

def inverse_metric_simpler_controllers_contraction_conditions(partial_f_x_partial_x,B,K,W, directional_f_W, directional_B_W, partial_B_partial_x,B_null, lambda_, **kwargs):
    """
    Returns 
    ```
    [
    -directional_f_W + (partial_f_x_partial_x + B@K)@W + (partial_f_x_partial_x + B@K).T@W + 2 * lambda_ * W,
    
    B_null@(-directional_f_W + partial_f_x_partial_x@W + (partial_f_x_partial_x@W).T + 2 * lambda_ * W)@B_null.T,
    
    partial_b_i_W - (partial_b_i_partial_x@W) - (partial_b_i_partial_x@W).T for b_i in columns of B
    ]
    ```
    """
    dfdxW = partial_f_x_partial_x@W
    C1_2 = -directional_f_W + dfdxW + dfdxW.T + 2 * lambda_ * W
    BKW = B@K@W
    # C1_1 = (-W_dot + ABKW.T + ABKW + 2 * lambda_ * W) but with C2 holding.
    C1_1 = C1_2 + BKW + BKW.T 
    # C1_2 = C1_1 for non actuable directions
    C1_2 = B_null@C1_2@B_null.T
    
    C2s = []
    for i in range(0,B.shape[1]):
        b_i = B[:,i]
        partial_b_i_partial_x = partial_B_partial_x[:,i,:]
        partial_b_i_W = directional_B_W[:,:,i]
        temp = partial_b_i_partial_x@W
        C2_condition_i = (partial_b_i_W - temp - temp.T)
        C2s.append(C2_condition_i)
    return C1_1,C1_2,C2s


In [ ]:
def calculate_common_terms(x,x_ref,u_ref):
    x.requires_grad = True
    f = dyn_sys.f_x(states = x)        
    K, u = torch.func.jacrev(lambda x: (controller(x,x_ref,u_ref),)*2, has_aux=True)(x)
    K = K.view(-1,num_states)
    partial_B_partial_x, B =  torch.func.jacrev(lambda x: (dyn_sys.B(states = x),)*2, has_aux=True)(x)
    partial_f_x_partial_x = dyn_sys.partial_f_x_partial_x(states = x)
    # A = dyn_sys.A(states = x, u = u)

    if dyn_sys.B_null:
        B_null = dyn_sys.B_null(x)
    else:
        U, S, Vh = torch.linalg.svd(B.T)
        B_null = Vh[dyn_sys.num_inputs:,:] # null space of size at least num_states - num_inputs
        
    delta_W_delta_x, W = torch.func.jacrev(lambda x: (contraction_metric(x),)*2, has_aux=True)(x)
    # M = torch.inverse(W)
    directional_f_W = (delta_W_delta_x @ f).squeeze()
    directional_B_W = (delta_W_delta_x @ B).squeeze()
    # W_dot = (delta_W_delta_x @ (f + B@(u.view(self.num_inputs,1)))).squeeze() 